In [ ]:
import sys
print(sys.executable)
#%pip install --upgrade pip
#%pip install torch torchvision torchaudio
%pip install -U pip
%pip install -U datasets transformers peft accelerate safetensors sentencepiece



In [8]:
import torch
print(torch.__version__)
#print(torch.version.cuda)
#print(torch.cuda.is_available())
#print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")
print("mps available: ", torch.backends.mps.is_available())
print("cuda available:", torch.cuda.is_available())
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device =", device)
import json
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainerCallback, EarlyStoppingCallback, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
import os, shutil

2.11.0
mps available:  True
cuda available: False
device = mps


In [9]:
import socket
import struct
import numpy as np

MAGIC = b"DFL1"
VERSION = 1
MSG_TRAIN_RESULT = 1
MSG_AGGREGATED_PARAMS = 2
MSG_SHUTDOWN = 3

def get_round_params(model, max_values=4):
    vals = []
    for name, p in model.named_parameters():
        if p.requires_grad:
            arr = p.detach().float().cpu().view(-1).numpy()
            need = max_values - len(vals)
            if need <= 0:
                break
            vals.extend(arr[:need].tolist())
    if len(vals) < max_values:
        vals.extend([0.0] * (max_values - len(vals)))
    
    return vals, 0.0

def send_train_result(sock, round_id, weights, bias):
    payload = struct.pack("<i", len(weights))
    payload += struct.pack("<" + "f" * len(weights), *[float(x) for x in weights])
    payload += struct.pack("<f", float(bias))

    header = struct.pack(">4sBBii", MAGIC, VERSION, MSG_TRAIN_RESULT, int(round_id), len(payload))
    sock.sendall(header + payload)

def recv_exact(sock, n):
    buf = bytearray()
    while len(buf) < n:
        chunk = sock.recv(n-len(buf))
        if not chunk:
            raise ConnectionError("Socket closed while receiving")
        buf.extend(chunk)
    return bytes(buf)

def recv_frame(sock):
    # Header: magic(4), version(1), type(1), round(4), payload_len(4) big-endian
    header = recv_exact(sock, 14)
    magic, version, msg_type, round_id, payload_len = struct.unpack(">4sBBii", header)

    if magic != MAGIC:
        raise ValueError(f"Bad magic: {magic}")
    if version != VERSION:
        raise ValueError(f"Bad version: {version}")
    if payload_len <= 0:
        raise ValueError(f"Bad payload length: {payload_len}")
    
    payload = recv_exact(sock, payload_len)

    # Payload: count(int32 LE), weights(float32 * count), bias(float32)
    count = struct.unpack_from("<i", payload, 0)[0]
    off = 4
    weights = list(struct.unpack_from("<" + "f" * count, payload, off))
    off += 4 * count
    bias = struct.unpack_from("<f", payload, off)[0]

    return msg_type, round_id, weights, bias


In [13]:
def main():
    # 1. Load Dataset
    #print("Loading dataset from local file medical_o1_sft_mix.json...")
    print("Loading small test dataset from dataset_part_1.jsonl and dataset_part_2.jsonl...")

    #dataset = load_dataset("json", data_files="medical_o1_sft_mix.json", split="train")
    dataset = load_dataset(
    "json",
    data_files=["dataset_part_1.jsonl", "dataset_part_2.jsonl"],
    split="train")
    
    dataset = dataset.shuffle(seed=42).select(range(min(len(dataset), 64)))
    print("Small dataset size:", len(dataset))
    

    split_1 = dataset.train_test_split(test_size=0.2, seed=42)
    split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)
    dataset = DatasetDict({
        "train": split_1["train"],
        "validation": split_2["train"],
        "test": split_2["test"],
    })
    print("Split sizes:", {k: len(dataset[k]) for k in dataset.keys()})

    # Format into Hugging Face Dataset format
    def process_messages(item):
        prompt = item["Question"]

        # Reasoning style answer combining CoT and final response
        cot = item.get('Complex_CoT', '') or ""
        response = item.get('Response', '') or ""
        answer = f"<think>\n{cot}\n</think>\n\n{response}"

        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": answer}
        ]
        return {"messages": messages}

    dataset = dataset.map(process_messages)

    # 2. Load Model and Tokenizer from HuggingFace
    model_name = "distilbert/distilgpt2"
    print(f"Loading {model_name} from HuggingFace Hub...")

    # Optional: If the model is gated or private, uncomment the following line and log in:
    # from huggingface_hub import login
    # login(token="HF_TOKEN")
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    def tokenize_function(examples):
        texts = []
        for msgs in examples["messages"]:
            try:
                text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            except Exception:
                text = f"<|im_start|>user\n{msgs[0]['content']}<|im_end|>\n<|im_start|>assistant\n{msgs[1]['content']}<|im_end|>"
            texts.append(text)

        encodings = tokenizer(texts, truncation=True, max_length=1024)
        return encodings   

    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=dataset["train"].column_names
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        #device_map="auto",
        #torch_dtype=torch.bfloat16,
        trust_remote_code=True
    )

    # 3. Setup DoRA
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["c_attn", "c_proj"],
        use_dora=False  # DoRA (Weight-Decomposed Low-Rank Adaptation)
    )

    model = get_peft_model(model, lora_config, autocast_adapter_dtype=False)
    _orig_load_adapter = model.load_adapter
    def _load_adapter_no_autocast(*args, **kwargs):
        kwargs.setdefault("autocast_adapter_dtype", False)
        return _orig_load_adapter(*args, **kwargs)
    model.load_adapter = _load_adapter_no_autocast
    model.print_trainable_parameters()

    # 4. Data Collator
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8)

    class KeepBestNCheckpointsCallback(TrainerCallback):
        def __init__(self, output_dir: str, n_best: int = 5):
            self.output_dir = output_dir
            self.n_best = n_best
            self.checkpoint_scores = {}  # checkpoint_dir_name -> eval_loss

        def on_evaluate(self, args, state, control, metrics=None, **kwargs):
            if not metrics:
                return control
            eval_loss = metrics.get("eval_loss")
            if eval_loss is None:
                return control
            ckpt_name = f"checkpoint-{state.global_step}"
            self.checkpoint_scores[ckpt_name] = float(eval_loss)
            return control

        def on_save(self, args, state, control, **kwargs):
            if not self.output_dir or not os.path.isdir(self.output_dir):
                return control

            ckpt_dirs = []
            for name in os.listdir(self.output_dir):
                full_path = os.path.join(self.output_dir, name)
                if name.startswith("checkpoint-") and os.path.isdir(full_path):
                    ckpt_dirs.append(name)

            if len(ckpt_dirs) <= self.n_best:
                return control

            def score(name: str) -> float:
                # Unknown scores get treated as worst so they are deleted first
                return self.checkpoint_scores.get(name, float("inf"))

            ckpt_dirs_sorted = sorted(ckpt_dirs, key=score)
            keep = set(ckpt_dirs_sorted[: self.n_best])

            for name in ckpt_dirs:
                if name in keep:
                    continue
                full_path = os.path.join(self.output_dir, name)
                try:
                    shutil.rmtree(full_path)
                except Exception:
                    pass

            return control

    # 5. Training Arguments
    training_args = TrainingArguments(
        # output_dir="./full_gpt2_lora",
        # per_device_train_batch_size=2,
        # gradient_accumulation_steps=4,
        # learning_rate=2e-4,
        # logging_steps=10,
        # num_train_epochs=3,
        # save_strategy="steps",
        # save_steps=100,
        # save_total_limit=None,
        # eval_strategy="steps",
        # eval_steps=100,
        # load_best_model_at_end=True,
        # metric_for_best_model="eval_loss",
        # greater_is_better=False,
        # bf16=True,
        # optim="adamw_torch",
        # report_to="none",
        # gradient_checkpointing=True
        output_dir="./full_gpt2_lora",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        learning_rate=2e-4,
        logging_strategy="epoch",
        logging_steps=1,
        num_train_epochs=3,      # 1 round = 3 epochs (your plan)
        save_strategy="no",      # first test: connection first
        eval_strategy="no",      # first test: connection first
        bf16=False,
        fp16=False,
        optim="adamw_torch",
        report_to="none",
        gradient_checkpointing=False
    )

    # 6. Initialize Trainer
    trainer = Trainer(
        # model=model,
        # train_dataset=tokenized_dataset["train"],
        # eval_dataset=tokenized_dataset["validation"],
        # args=training_args,
        # data_collator=data_collator,
        # callbacks=[
        #     KeepBestNCheckpointsCallback(output_dir=training_args.output_dir, n_best=5),
        #     EarlyStoppingCallback(early_stopping_patience=3, early_stopping_thresh=1e-5),
        # ],

        model=model,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        args=training_args,
        data_collator=data_collator
    )

    # 7. Start Training
    print("Starting baseline finetuning for full dataset...")

    # Take care of this
    HOST = "127.0.0.1"
    PORT = 5001
    TOTAL_ROUNDS = 3   # must match Java totalRounds

    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

    server.bind((HOST, PORT))
    server.listen(1)

    print("Python listening on 127.0.0.1:5001 ...")

    conn, addr = server.accept()
    print("Java connected from:", addr)


    for round_id in range(1, TOTAL_ROUNDS + 1):

        print(f"\n=== Python Round {round_id} ===")

        trainer.train()

        weights, bias = get_round_params(model, max_values=4)

        send_train_result(conn, round_id, weights, bias)

        msg_type, rr, agg_w, agg_b = recv_frame(conn)

        if msg_type == MSG_SHUTDOWN:
            print("Received SHUTDOWN from Java")
            break

        if msg_type != MSG_AGGREGATED_PARAMS:
            raise ValueError(f"Expected AGGREGATED_PARAMS, got type={msg_type}")

        if rr != round_id:
            raise ValueError(
                f"Round mismatch: expected {round_id}, got {rr}"
            )

        print(
            f"Round {rr} success: received {len(agg_w)} weights"
        )


    conn.close()
    server.close()
    # Take care of the above

    # trainer.train()
    # weights, bias = get_round_params(model, max_values=4)

    # with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server:
    #     server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    #     server.bind(("127.0.0.1", 9001))
    #     server.listen(1)
    #     print("Python listening on 127.0.0.1:9001 ...")

    #     conn, addr = server.accept()
    #     with conn:
    #         print("Java connected from:", addr)

    #         send_train_result(conn, round_id=1, weights=weights, bias=bias)

    #         msg_type, rr, agg_w, agg_b = recv_frame(conn)
    #         if msg_type == MSG_SHUTDOWN:
    #             print("Received SHUTDOWN from Java")
    #         elif msg_type != MSG_AGGREGATED_PARAMS:
    #             raise ValueError(f"Expected AGGREGATED_PARAMS, got type={msg_type}")
    #         elif rr != 1:
    #             raise ValueError(f"Round mismatch. expected=1 got={rr}")
    #         else:
    #             print(f"Receive success: round={rr}, n_weights={len(agg_w)}, bias={agg_b}")

    import math
    try:
        from transformers.utils.notebook import NotebookProgressCallback
        trainer.remove_callback(NotebookProgressCallback)
    except Exception:
        pass
    test_metrics = trainer.evaluate(eval_dataset=tokenized_dataset["test"])
    print("Test metrics:", test_metrics)
    if "eval_loss" in test_metrics and test_metrics["eval_loss"] is not None:
        print("Test perplexity:", math.exp(test_metrics["eval_loss"]))

    # 8. Save final model
    print("Saving model...")
    trainer.model.save_pretrained("./full_gpt2_lora_final")
    tokenizer.save_pretrained("./full_gpt2_lora_final")
    print("Done!")

In [14]:
if __name__ == "__main__":
    main()

Loading small test dataset from dataset_part_1.jsonl and dataset_part_2.jsonl...
Small dataset size: 64
Split sizes: {'train': 51, 'validation': 6, 'test': 7}
Loading distilbert/distilgpt2 from HuggingFace Hub...


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 8687.65it/s]
/Users/DELL/Documents/3rd Year/2nd_sem/Distributed_System/DistriSys-DFL-Hospital/.venv/lib/python3.14/site-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 811,008 || all params: 82,723,584 || trainable%: 0.9804
Starting baseline finetuning for full dataset...
Python listening on 127.0.0.1:5001 ...
Java connected from: ('127.0.0.1', 61812)

=== Python Round 1 ===


/Users/DELL/Documents/3rd Year/2nd_sem/Distributed_System/DistriSys-DFL-Hospital/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
51,3.155947
102,3.009553
153,2.951333


Round 1 success: received 4 weights

=== Python Round 2 ===


Step,Training Loss
51,2.888299
102,2.826360
153,2.790236


Received SHUTDOWN from Java
Test metrics: {'eval_loss': 2.7680132389068604, 'eval_runtime': 0.7048, 'eval_samples_per_second': 9.932, 'eval_steps_per_second': 1.419, 'epoch': 3.0}
Test perplexity: 15.926959491912456
Saving model...
Done!
